```sql
DROP TABLE IF EXISTS clean_telecom_data;

WITH Deduped AS (
    SELECT
        customerID,
        gender,
        SeniorCitizen,
        Partner,
        Dependents,
        CAST(tenure AS INT) AS tenure,

        PhoneService,
        CASE WHEN MultipleLines = 'No phone service' THEN 'No' ELSE MultipleLines END AS MultipleLines,
        InternetService,
        CASE WHEN OnlineSecurity   = 'No internet service' THEN 'No' ELSE OnlineSecurity   END AS OnlineSecurity,
        CASE WHEN OnlineBackup     = 'No internet service' THEN 'No' ELSE OnlineBackup     END AS OnlineBackup,
        CASE WHEN DeviceProtection = 'No internet service' THEN 'No' ELSE DeviceProtection END AS DeviceProtection,
        CASE WHEN TechSupport      = 'No internet service' THEN 'No' ELSE TechSupport      END AS TechSupport,
        CASE WHEN StreamingTV      = 'No internet service' THEN 'No' ELSE StreamingTV      END AS StreamingTV,
        CASE WHEN StreamingMovies  = 'No internet service' THEN 'No' ELSE StreamingMovies  END AS StreamingMovies,

        Contract,
        PaperlessBilling,
        PaymentMethod,

        CAST(MonthlyCharges AS DECIMAL(10,2)) AS MonthlyCharges,

        CAST(
            COALESCE(NULLIF(TRIM(TotalCharges), ''), '0')
            AS DECIMAL(10,2)
        ) AS TotalCharges,

        Churn,
        ROW_NUMBER() OVER (PARTITION BY customerID ORDER BY customerID) AS rn

    FROM raw_telecom_data
)
SELECT
    customerID, gender, SeniorCitizen, Partner, Dependents, tenure,
    PhoneService, MultipleLines, InternetService, OnlineSecurity,
    OnlineBackup, DeviceProtection, TechSupport, StreamingTV,
    StreamingMovies, Contract, PaperlessBilling, PaymentMethod,
    MonthlyCharges, TotalCharges, Churn
INTO clean_telecom_data
FROM Deduped
WHERE rn = 1;
``` 

-- =========================================================
-- KPI 1: Overall Retention Rate (%)
-- =========================================================
SELECT
    COUNT(CASE WHEN Churn = 'No' THEN 1 END) AS ActiveCustomers,
    COUNT(*) AS TotalCustomers,
    CAST(
        COUNT(CASE WHEN Churn = 'No' THEN 1 END) * 100.0 / NULLIF(COUNT(*), 0)
    AS DECIMAL(5,2)) AS RetentionRatePct
FROM clean_telecom_data;


-- =========================================================
-- KPI 2: Averages Breakdown (MonthlyCharges, TotalCharges, Tenure)
-- =========================================================
SELECT
    CAST(AVG(MonthlyCharges) AS DECIMAL(10,2)) AS AvgMonthlyCharges,
    CAST(AVG(TotalCharges)   AS DECIMAL(10,2)) AS AvgTotalCharges,
    CAST(AVG(CAST(tenure AS FLOAT)) AS DECIMAL(10,2)) AS AvgTenureMonths
FROM clean_telecom_data;


-- =========================================================
-- KPI 3: Contract Distribution (Count + % of Total)
-- =========================================================
SELECT
    Contract,
    COUNT(*) AS CustomerCount,
    CAST(
        COUNT(*) * 100.0 / NULLIF(SUM(COUNT(*)) OVER (), 0)
    AS DECIMAL(5,2)) AS PctOfTotal
FROM clean_telecom_data
GROUP BY Contract
ORDER BY CustomerCount DESC;

-- =========================================================
-- KPI 1: Overall Retention Rate (%)
-- =========================================================
SELECT
    COUNT(CASE WHEN Churn = 'No' THEN 1 END) AS ActiveCustomers,
    COUNT(*) AS TotalCustomers,
    CAST(
        COUNT(CASE WHEN Churn = 'No' THEN 1 END) * 100.0 / NULLIF(COUNT(*), 0)
    AS DECIMAL(5,2)) AS RetentionRatePct
FROM clean_telecom_data;


-- =========================================================
-- KPI 2: Averages Breakdown (MonthlyCharges, TotalCharges, Tenure)
-- =========================================================
SELECT
    CAST(AVG(MonthlyCharges) AS DECIMAL(10,2)) AS AvgMonthlyCharges,
    CAST(AVG(TotalCharges)   AS DECIMAL(10,2)) AS AvgTotalCharges,
    CAST(AVG(CAST(tenure AS FLOAT)) AS DECIMAL(10,2)) AS AvgTenureMonths
FROM clean_telecom_data;


-- =========================================================
-- KPI 3: Contract Distribution (Count + % of Total)
-- =========================================================
SELECT
    Contract,
    COUNT(*) AS CustomerCount,
    CAST(
        COUNT(*) * 100.0 / NULLIF(SUM(COUNT(*)) OVER (), 0)
    AS DECIMAL(5,2)) AS PctOfTotal
FROM clean_telecom_data
GROUP BY Contract
ORDER BY CustomerCount DESC;